# 03 - Train / Validation / Test Split

In this notebook, the target variable is separated from the input features, potential leakage columns are removed, and the dataset is divided into training, validation, and test sets.

A **temporal split** is used to preserve the chronological order of the orders and better simulate the production scenario, where a model trained on historical data is used to make predictions on future orders.

Special attention is given to preventing data leakage and ensuring that only information available at prediction time is included in the model features.

## Import Libraries and Data

In [ ]:
import pandas as pd

In [2]:
data = pd.read_csv("data/labeled_orders.csv")
print("Dataset shape:", data.shape)
data.head()

Dataset shape: (96470, 29)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,seller_count,seller_state_count,seller_states,seller_zip_code_prefix,customer_lat,customer_lng,seller_lat,seller_lng,distance_km,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,1.0,SP,9350.0,-23.576983,-46.587161,-23.680729,-46.444238,18.576110,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,SP,31570.0,-12.177924,-44.660711,-19.807681,-43.980427,851.495069,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,SP,14840.0,-16.745150,-48.514783,-21.363502,-48.229601,514.410666,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,MG,31842.0,-5.774190,-35.271143,-19.837682,-43.924053,1822.226336,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,1.0,SP,8752.0,-23.676370,-46.514627,-23.543395,-46.262086,29.676625,0


## Feature and Target Inspection

Before splitting the dataset, we inspect all available columns to identify the target variable and any features that may cause data leakage.

The target variable is is_late, where:
- 0 = the order was delivered on time.
- 1 = the order was delivered late.

In [3]:
data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_price',
 'total_freight',
 'item_count',
 'total_payment',
 'payment_installments',
 'payment_count',
 'review_score',
 'seller_count',
 'seller_state_count',
 'seller_states',
 'seller_zip_code_prefix',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng',
 'distance_km',
 'is_late']

## Data Leakage Prevention

Some columns contain information generated after the order was placed or delivered. Using these features could cause data leakage and produce unrealistically high model performance.

Potential leakage features include:

- order_delivered_customer_date: actual delivery date and directly related to the target definition.
- order_delivered_carrier_date: becomes available after the order progresses.
- review_score: generated after the customer receives and reviews the order.
- order_status: may contain information about events occurring after purchase.

The target column is_late is also excluded from the input features because it is the target that the model will learn to predict.


* Identifier columns are also excluded because they uniquely identify orders or customers rather than representing meaningful predictive features

_______
## Prediction Time

The prediction is assumed to be made **after the order is created and approved**, but **before the order is shipped or delivered**.

Therefore, features available at or shortly after purchase can be used, while information generated during or after delivery must be excluded.


In [4]:
y = data['is_late']
print("Target Shape:", y.shape)
print("Target distribution:")
print(y.value_counts())


Target Shape: (96470,)
Target distribution:
is_late
0    88644
1     7826
Name: count, dtype: int64


In [5]:
print("\nTarget percentages:")
print(y.value_counts(normalize=True).mul(100).round(2))


Target percentages:
is_late
0    91.89
1     8.11
Name: proportion, dtype: float64


## Feature Selection

- The target variable is_late is y

- The remained features are X

In [6]:
columns_to_drop = [
    "is_late",                       # target
    "order_id",                      # id
    "customer_id",                   # id
    "customer_unique_id",            # id
    "order_status",                  # may contain future information
    "order_delivered_carrier_date",  # known after shipping
    "order_delivered_customer_date", # actual delivery -> direct leakage
    "review_score"                   # known after delivery
]

In [7]:
X = data.drop(columns=columns_to_drop)

In [8]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X.columns.tolist()

X shape: (96470, 21)
y shape: (96470,)


['order_purchase_timestamp',
 'order_approved_at',
 'order_estimated_delivery_date',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_price',
 'total_freight',
 'item_count',
 'total_payment',
 'payment_installments',
 'payment_count',
 'seller_count',
 'seller_state_count',
 'seller_states',
 'seller_zip_code_prefix',
 'customer_lat',
 'customer_lng',
 'seller_lat',
 'seller_lng',
 'distance_km']

## Train / Validation / Test Split

The dataset is divided chronologically into three subsets based on `order_purchase_timestamp`:

- **Training set (70%)** → the oldest orders, used to train the machine learning model.
- **Validation set (15%)** → the next period of orders, used for model selection and hyperparameter tuning.
- **Test set (15%)** → the most recent orders, kept unseen until the final evaluation.

A **temporal (chronological) split** is used instead of a random stratified split.

This better reflects the intended production scenario, where the model is trained on historical orders and then used to predict whether future orders will be delivered late.

The chronological order is preserved, so no future orders are included in the training set.

In [9]:
purchase_dates = pd.to_datetime(data["order_purchase_timestamp"])

print("First order:", purchase_dates.min())
print("Last order:", purchase_dates.max())

First order: 2016-09-15 12:16:38
Last order: 2018-08-29 15:00:37


In [10]:
# Sort all orders chronologically from oldest to newest
sorted_idx = pd.to_datetime(
    data["order_purchase_timestamp"]
).sort_values().index

X_sorted = X.loc[sorted_idx].copy()
y_sorted = y.loc[sorted_idx].copy()

# Calculate split boundaries
n = len(X_sorted)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Oldest 70% -> Training set
X_train = X_sorted.iloc[:train_end].copy()
y_train = y_sorted.iloc[:train_end].copy()

# Next 15% -> Validation set
X_val = X_sorted.iloc[train_end:val_end].copy()
y_val = y_sorted.iloc[train_end:val_end].copy()

# Newest 15% -> Test set
X_test = X_sorted.iloc[val_end:].copy()
y_test = y_sorted.iloc[val_end:].copy()

### Split Strategy

A **time-based (temporal) split** is used to simulate the real production scenario.

All orders are first sorted chronologically by `order_purchase_timestamp`, from the oldest to the most recent.

The ordered dataset is then divided as follows:

- Oldest 70% → Training set
- Next 15% → Validation set
- Newest 15% → Test set

Unlike random stratified splitting, the data is not shuffled and stratification is not applied because preserving the chronological order is more important for evaluating performance on future orders.

As a result, the late-delivery rate may differ between the three subsets. These differences are intentionally preserved because they may reflect real changes in the data distribution over time.

_____

### Verify Class Distribution After Temporal Splitting

The original target is imbalanced, with most orders delivered on time and a smaller proportion delivered late.

Because a temporal split is used, stratification is not applied. Therefore, the proportion of late deliveries is not expected to be identical across the training, validation, and test sets.

The observed late-delivery rates are:

- **Training:** 9.03%
- **Validation:** 5.34%
- **Test:** 6.61%

These differences are preserved intentionally because they may represent real changes in the late-delivery rate over time.

All three subsets still contain examples from both target classes, allowing the model to be trained, validated, and tested appropriately.

In [11]:
train_data = X_train.copy()
train_data["is_late"] = y_train

val_data = X_val.copy()
val_data["is_late"] = y_val

test_data = X_test.copy()
test_data["is_late"] = y_test

In [12]:
train_order_ids = data.loc[X_train.index, ["order_id"]].copy()

print(train_order_ids.shape)
train_order_ids.head()

(67529, 1)


,order_id
29809,bfbd0f9bdef84302105ad712db648a6c
90504,3b697a20d9e427646d92567910af6d57
27597,be5bc2f0da14d8071e2d45451ad119d9
95052,a41c8759fbe7aab36ea07e038b2d4465
85825,d207cc272675637bfed0062edffd0818


### Verify Chronological Separation

To confirm that the temporal split was applied correctly, the date range of each subset is checked.

The training period should occur before the validation period, and the validation period should occur before the test period.

In [13]:
for name, df in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    dates = pd.to_datetime(df["order_purchase_timestamp"])
    print(
        name,
        "|", dates.min(),
        "to", dates.max()
    )

Train | 2016-09-15 12:16:38 to 2018-04-15 20:12:35
Validation | 2018-04-15 20:17:11 to 2018-06-21 08:29:29
Test | 2018-06-21 08:41:07 to 2018-08-29 15:00:37


In [14]:
train_max = pd.to_datetime(train_data["order_purchase_timestamp"]).max()
val_min = pd.to_datetime(val_data["order_purchase_timestamp"]).min()
val_max = pd.to_datetime(val_data["order_purchase_timestamp"]).max()
test_min = pd.to_datetime(test_data["order_purchase_timestamp"]).min()

print("Train -> Validation chronological:",train_max <= val_min)

print("Validation -> Test chronological:",val_max <= test_min)

Train -> Validation chronological: True
Validation -> Test chronological: True


In [15]:
print("Train target distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).mul(100).round(2))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Train target distribution:
is_late
0    90.97
1     9.03
Name: proportion, dtype: float64

Validation target distribution:
is_late
0    94.66
1     5.34
Name: proportion, dtype: float64

Test target distribution:
is_late
0    93.39
1     6.61
Name: proportion, dtype: float64


In [16]:
train_data.to_csv("data/train.csv", index=False)
val_data.to_csv("data/validation.csv", index=False)
test_data.to_csv("data/test.csv", index=False)

print("Data splits saved successfully.")

Data splits saved successfully.


In [17]:
train_order_ids.to_csv("data/train_order_ids.csv", index=False)

print("Train order IDs saved successfully.")

Train order IDs saved successfully.
